# EverFlavor AI - Dish Knowledge: the Waiter's Notebook

**Team:** EverGlow | **Course:** ITAI 2277 | **Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: new for 0.6.3 (2026-11-10).** Run it cell by cell. Section 2 (collecting) is off by default: the
> collected file is in the repository.

---

## Purpose

The waiter (0.6.3) tells a dish's story, fun facts and history. Nobody writes those by hand: they are collected
from open sources that already hold them, and this notebook tidies them into a readable knowledge notebook.

| Section | What it does | Output |
|---|---|---|
| 2 | Collects the dishes of the countries and regions the app shows (Wikipedia lists and intros, Wikidata) | `data/knowledge/dishes.jsonl` |
| 3 | Tidies the fun facts (translations in brackets removed, "It is ..." given the dish's name, company news dropped) and sorts them by topic: inspiration, origin, culture, history, about. Three from different topics come first; the rest wait as "more fun facts" for a user who asks | in memory |
| 4 | Writes one page per country, with sources and "learn more" links by interest level | `data/knowledge/dishes/` |

**Rules:** facts are the sources' words (nothing invented); Wikipedia text is CC BY-SA 4.0, so every dish keeps
its credit and link; the cleaned facts are marked `auto-cleaned` until a person has read them.

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [ ]:
# %pip (not !pip) installs into the Python that runs this notebook
%pip install -q pandas pyarrow requests

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "cooking", "cooking_time", "dish_steps", "crew",
                      "cuisine", "diets", "dish_knowledge", "dish_names", "environment", "evaluation", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline",
                      "progress", "recommend", "refresh", "reporting", "review", "safety", "seasons", "sources",
                      "saved_recipes", "stores", "variants", "waiter"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

EVAL_DIR = Path("data/processed/agent_eval")   # one CSV per run, compared with the last one
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")

### Settings

`COLLECT` reruns the collector with `--stories` (each article's History, Origin and Culture sections too: about
2 hours the first time, minutes with its cache in `data/raw/knowledge_cache/`). Leave it off to use the collected file.

In [ ]:
COLLECT = False
KNOWLEDGE = PROJECT_ROOT / "data" / "knowledge"

---

## 2. Collect (optional)

In [ ]:
import subprocess

if COLLECT:
    subprocess.run([sys.executable, "scripts/knowledge/fetch_knowledge.py", "--stories"], check=True)
else:
    print("Using the collected file:", KNOWLEDGE / "dishes.jsonl")

---

## 3. Tidy the fun facts

In [ ]:
from everflavor import dish_knowledge as dk

raw = dk.load_dishes(KNOWLEDGE / "dishes.jsonl")
dishes = [dk.tidy_dish(d) for d in raw]
table = pd.DataFrame(dishes)
print(f"{len(table):,} entries, {(table['kind'] == 'dish').sum():,} dishes")
print(f"facts kept: {table['fun_facts'].str.len().sum():,} shown first, {table['more_fun_facts'].str.len().sum():,} more on request")
print("topics:", pd.Series([f["topic"] for fs in table["fun_facts"] + table["more_fun_facts"] for f in fs]).value_counts().to_dict())
print(f"dishes with at least one fun fact: {(table['fun_facts'].str.len() > 0).sum():,}")
table.groupby("country").agg(entries=("name", "size"), with_facts=("fun_facts", lambda s: (s.str.len() > 0).sum())).sort_values("entries", ascending=False).head(15)

A few before-and-after examples (the source's sentence, then what the waiter may say):

In [ ]:
changed = [(d["name"], c, dk.clean_fact(c, d["name"].split(" (")[0])) for d in raw for c in d["fun_fact_candidates"]
           if dk.clean_fact(c, d["name"].split(" (")[0]) != c]
print(f"{len(changed):,} sentences changed by the rules")
for name, before, after in changed[:5]:
    print(name)
    print("  before:", before)
    print("  after: ", after)


---

## 4. Write the knowledge notebook

In [ ]:
counts = dk.write_notebook(dishes, KNOWLEDGE / "dishes")
print(f"{len(counts)} country pages in {KNOWLEDGE / 'dishes'}")
print(dk.dish_card(next(d for d in dishes if d["fun_facts"] and d.get("region"))))

---

## 5. Next

- A person reads the `auto-cleaned` facts of the focus areas first (USA, Mexico, China, Middle East, Africa) and
  marks them reviewed; teammates may add their own, each with a source.
- Thin spots: China's list links few dishes (add its regional cuisine pages); Mexico and Africa need divisions
  (Wikidata origin plus the division map).